# 10 — Hydroclimatic controls of preliminary Sentinel-1 inundation signals (2019–2025)

**Hakaluki Haor | offline, reproducible exploratory analysis.** Integrates Notebook 09's QA-revised monthly low-VV areas with previously extracted CHIRPS monthly rainfall and ERA5-Land temperature. **No Google Earth Engine calls or satellite downloads are needed.**

**Scientific boundary:** Low-VV (`VV ≤ −17 dB`) is a radar-screening indicator, **not** a validated flooded-area estimate. Correlation and year-held-out prediction do **not** establish causal rainfall effects or methane emissions. Thresholds (`−19/−17/−15 dB`), seasonal adjustment, and gaps are reported.

**Design:** strict date alignment → missingness and provenance audit → month-matched departures and rainfall lags → descriptive, deseasonalized associations → leave-one-year-out (LOYO) seasonal-baseline versus climate-enhanced model → automated figures, CSVs and provisional research brief. Rainfall lags are antecedent measured rainfall, not future forecasts.

## 10A — Read the real files and verify time alignment

Expected existing datasets:

- `results/tables/inundation09_monthly_2019_2025.csv` (Notebook 09 QA v2/v3, 84 months, single ascending orbit 41)
- `data/processed/hakaluki_monthly_2019_2025.csv` (previous CHIRPS and ERA5-Land extraction)

If your climate CSV uses other column names, change only `RAIN_COL` and `TEMP_COL` below **after inspecting the printed headers**; never silently guess units. No internal notebook variables from Notebook 09 are required.

In [ ]:
from pathlib import Path
import json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(r"F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project")
S1_CSV = ROOT / "results/tables/inundation09_monthly_2019_2025.csv"
CLIMATE_CSV = ROOT / "data/processed/hakaluki_monthly_2019_2025.csv"
PARAMS_JSON = ROOT / "results/tables/inundation09_processing_parameters.json"
TABLES = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"
REPORTS = ROOT / "results/reports"
for directory in (TABLES, FIGURES, REPORTS):
    directory.mkdir(parents=True, exist_ok=True)

RAIN_COL = "rain_chirps_mm"       # monthly total, mm; polygon-mean CHIRPS
TEMP_COL = "temp2m_era5_c"       # ERA5-Land monthly 2 m temperature, Celsius
TARGET_COL = "nominal_lowvv_km2" # 2019–2025 screening area from Notebook 09
MIN_COVERAGE = 0.90
SEED = 20261014

for required in (S1_CSV, CLIMATE_CSV, PARAMS_JSON):
    if not required.is_file():
        raise FileNotFoundError(f"Required input is missing: {required}")

s1_raw = pd.read_csv(S1_CSV)
clim_raw = pd.read_csv(CLIMATE_CSV)
parameters = json.loads(PARAMS_JSON.read_text(encoding="utf-8"))

print("=== ORIGINAL DATASET COLUMNS ===")
print("Notebook 09:", s1_raw.columns.tolist())
print("Hydroclimate:", clim_raw.columns.tolist())
print("Config:", parameters.get("config_hash"))
print("Rows:", len(s1_raw), len(clim_raw))

def standardize_dates(frame, label):
    frame = frame.copy()
    if {"year", "month"}.issubset(frame.columns):
        yrs = pd.to_numeric(frame["year"], errors="raise").astype(int)
        mons = pd.to_numeric(frame["month"], errors="raise").astype(int)
        if not mons.between(1,12).all():
            raise ValueError(f"{label}: invalid month values")
        frame["date"] = pd.to_datetime(dict(year=yrs, month=mons, day=1))
    elif "period" in frame.columns:
        frame["date"] = pd.to_datetime(frame["period"].astype(str).str[:7] + "-01", errors="raise")
    elif "date" in frame.columns:
        frame["date"] = pd.to_datetime(frame["date"], errors="raise").dt.to_period("M").dt.to_timestamp()
    else:
        raise ValueError(f"{label}: cannot find year+month, period, or date")
    if frame["date"].duplicated().any():
        bad=frame.loc[frame["date"].duplicated(False), "date"].dt.strftime("%Y-%m").tolist()
        raise ValueError(f"{label}: duplicate year/month rows: {bad[:12]}")
    return frame

s1 = standardize_dates(s1_raw, "Notebook 09")
clim = standardize_dates(clim_raw, "Hydroclimate")

expected_dates = pd.date_range("2019-01-01", "2025-12-01", freq="MS")
if len(s1) != 84 or not s1["date"].sort_values().reset_index(drop=True).equals(pd.Series(expected_dates)):
    raise RuntimeError("Notebook 09 input must include exactly 84 unique months, Jan 2019–Dec 2025")

if not {RAIN_COL, TEMP_COL}.issubset(clim.columns):
    raise ValueError("Required CHIRPS/ERA5 columns missing; inspect printed headers and set RAIN_COL/TEMP_COL explicitly")

if "config_hash" not in s1.columns or s1["config_hash"].nunique() != 1:
    raise RuntimeError("Notebook 09 CSV lacks a single configuration fingerprint")
config_hash = str(s1["config_hash"].iloc[0])
if config_hash != str(parameters.get("config_hash")):
    raise RuntimeError("Processing fingerprints disagree: do not join mismatched Notebook 09 versions")
if "preserve_original_scene_mask" in parameters and not parameters["preserve_original_scene_mask"]:
    raise RuntimeError("Notebook 09 must use restored original valid-data support")

s1_needed = ["date", "year", "month", "scene_count", "observed_fraction", "comparison_eligible", "config_hash", "lowvv_m19_km2", "nominal_lowvv_km2", "lowvv_m15_km2"]
missing = [c for c in s1_needed if c not in s1.columns]
if missing:
    raise ValueError(f"Notebook 09 missing required fields: {missing}")

climate_part = clim[["date", RAIN_COL, TEMP_COL]].copy()
climate_part = climate_part.rename(columns={RAIN_COL: "rain_mm", TEMP_COL: "temperature_c"})
monthly10 = s1[s1_needed].merge(climate_part, on="date", how="left", validate="one_to_one", indicator=True)
monthly10 = monthly10.sort_values("date").reset_index(drop=True)
monthly10["rain_mm"] = pd.to_numeric(monthly10["rain_mm"], errors="coerce")
monthly10["temperature_c"] = pd.to_numeric(monthly10["temperature_c"], errors="coerce")
monthly10["comparison_eligible"] = monthly10["comparison_eligible"].astype(str).str.lower().isin(["true", "1"])
monthly10["observed_fraction"] = pd.to_numeric(monthly10["observed_fraction"], errors="coerce")
for c in ["nominal_lowvv_km2", "lowvv_m19_km2", "lowvv_m15_km2"]:
    monthly10[c] = pd.to_numeric(monthly10[c], errors="coerce")
monthly10["radar_ok"] = (monthly10["comparison_eligible"] & monthly10["observed_fraction"].ge(MIN_COVERAGE) & monthly10[TARGET_COL].notna())
monthly10["year"] = monthly10["date"].dt.year
monthly10["month"] = monthly10["date"].dt.month

if (monthly10["rain_mm"].dropna() < 0).any():
    raise ValueError("Negative CHIRPS monthly rainfall found; inspect input units/quality")
if monthly10["temperature_c"].dropna().between(0, 55).mean() < 0.95:
    warnings.warn("ERA5 temperature values appear unusual for Celsius; confirm original column and conversion")
if (monthly10["nominal_lowvv_km2"] > 180).any():
    warnings.warn("Low-VV area above expected ROI size; review masks and units")

print("\n=== INPUT ALIGNMENT ===")
print("Climate matched months:", int((monthly10["_merge"] == "both").sum()), "/84")
print("Radar-eligible months:", int(monthly10["radar_ok"].sum()), "/84")
print("Rain valid:", int(monthly10.rain_mm.notna().sum()), "/84")
print("Temperature valid:", int(monthly10.temperature_c.notna().sum()), "/84")
print("Selected config:", config_hash)
display(monthly10[["date", TARGET_COL, "rain_mm", "temperature_c", "scene_count", "radar_ok"]].head(12))

## 10B — Missingness and antecedent rainfall

Construct 0–3-month CHIRPS rainfall lags and a **strictly previous-three-month** precipitation sum (months *t−1, t−2, t−3*). Rolling predictors never use future months. Both the optical-independent CHIRPS and radar response must be date-aligned; monthly hydroclimate products are much coarser than the 30 m radar grid.

In [ ]:
# All 84 rows are retained. The first 1–3 lagged months are missing by definition.
for lag in range(4):
    monthly10[f"rain_lag{lag}_mm"] = monthly10["rain_mm"].shift(lag)
monthly10["antecedent_3m_rain_mm"] = monthly10["rain_mm"].shift(1).rolling(3, min_periods=3).sum()
monthly10["temperature_lag1_c"] = monthly10["temperature_c"].shift(1)
monthly10["threshold_span_km2"] = monthly10["lowvv_m15_km2"] - monthly10["lowvv_m19_km2"]
monthly10["rain_available"] = monthly10.rain_mm.notna()
monthly10["temp_available"] = monthly10.temperature_c.notna()
monthly10["joint_available"] = monthly10["radar_ok"] & monthly10["rain_available"] & monthly10["temp_available"]

print("=== 10B QUALITY AUDIT ===")
print("Joint radar/rain/temperature months:", int(monthly10.joint_available.sum()))
print("3-month antecedent rainfall available:", int(monthly10.antecedent_3m_rain_mm.notna().sum()))
print("Single-scene months:", int((monthly10.scene_count == 1).sum()))
print("Low-VV threshold span median (km²):", round(float(monthly10.threshold_span_km2.median()), 2))

qa_by_year = (monthly10.groupby("year", as_index=False).agg(
    months=("date", "size"), radar_eligible=("radar_ok", "sum"),
    rainfall_valid=("rain_available", "sum"), temperature_valid=("temp_available", "sum"),
    joint_valid=("joint_available", "sum"),
    annual_chirps_rain_mm=("rain_mm", lambda s: s.sum(min_count=1)),
    median_lowvv_km2=(TARGET_COL, "median"),
    median_threshold_span_km2=("threshold_span_km2", "median")))
display(qa_by_year)
monthly10.to_csv(TABLES / "hydro10_merged_monthly_2019_2025.csv", index=False)
qa_by_year.to_csv(TABLES / "hydro10_annual_input_qa.csv", index=False)

## 10C — Seasonal context and visual diagnostics

Plots preserve **measured** monthly totals, rather than downscaling them to fictitious 30 m rainfall grids. Comparisons among months must account for Bangladesh's strong monsoonal seasonality.

In [ ]:
plot_monthly = monthly10.copy()
climatology = (plot_monthly.groupby("month", as_index=False).agg(
    median_lowvv_km2=(TARGET_COL, "median"),
    median_rain_mm=("rain_mm", "median"),
    median_temp_c=("temperature_c", "median")))
climatology.to_csv(TABLES / "hydro10_monthly_seasonal_climatology.csv", index=False)

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True, constrained_layout=True)
axes[0].plot(monthly10.date, monthly10[TARGET_COL].where(monthly10.radar_ok), lw=1.35)
axes[0].fill_between(monthly10.date, monthly10.lowvv_m19_km2, monthly10.lowvv_m15_km2, alpha=.18)
axes[0].set_ylabel("Low-VV area (km²)")
axes[0].set_title("Sentinel-1 screening response (threshold sensitivity shown)")
axes[1].bar(monthly10.date, monthly10.rain_mm, width=23)
axes[1].set_ylabel("CHIRPS rainfall (mm/month)")
axes[2].plot(monthly10.date, monthly10.temperature_c, marker=".", lw=1.05)
axes[2].set_ylabel("ERA5-Land 2 m temp. (°C)")
axes[2].set_xlabel("Month (UTC)")
for ax in axes: ax.grid(alpha=.2)
fig.suptitle("Hakaluki Haor | Monthly hydroclimate and preliminary radar response, 2019–2025")
path_ts = FIGURES / "hydro10_timeseries_rain_temperature_lowvv.png"
fig.savefig(path_ts, dpi=230, bbox_inches="tight"); plt.show(); plt.close(fig)

fig, axs = plt.subplots(1, 3, figsize=(14, 4.6), constrained_layout=True)
for ax, col, lab in zip(axs,
        ["median_lowvv_km2", "median_rain_mm", "median_temp_c"],
        ["Low-VV area (km²)", "Rainfall (mm/month)", "Temperature (°C)"]):
    ax.plot(climatology.month, climatology[col], marker="o")
    ax.set_xticks(range(1, 13)); ax.set_xlabel("Calendar month")
    ax.set_ylabel(lab); ax.grid(alpha=.2)
fig.suptitle("Month-of-year medians, 2019–2025 | Seasonal context")
path_season = FIGURES / "hydro10_seasonal_cycle.png"
fig.savefig(path_season, dpi=230, bbox_inches="tight"); plt.show(); plt.close(fig)
print("Saved:", path_ts.name, "and", path_season.name)

## 10D — Leave-one-year-out calendar-month departures

For every year and month, the reference is the **median of the same calendar month in the other years**. This is a descriptive reference (six observations for complete seven-year variables), not an estimate of a counterfactual. The 2022 rainfall and radar departures can be compared without mistaking seasonal June rainfall for an anomaly.

In [ ]:
from scipy.stats import spearmanr, pearsonr

def other_year_month_median(frame, variable):
    """Reference for each row is the median of that calendar month in other years."""
    answers=[]
    for _, row in frame.iterrows():
        vals = frame.loc[(frame.month == row.month) & (frame.year != row.year), variable].dropna()
        answers.append(float(vals.median()) if len(vals) >= 3 else np.nan)
    return np.asarray(answers, dtype=float)

for metric in [TARGET_COL, "lowvv_m19_km2", "lowvv_m15_km2", "rain_mm", "temperature_c", "rain_lag1_mm", "rain_lag2_mm", "rain_lag3_mm", "antecedent_3m_rain_mm"]:
    refcol = f"{metric}_other_year_median"
    anomalycol = f"{metric}_departure"
    monthly10[refcol] = other_year_month_median(monthly10, metric)
    monthly10[anomalycol] = monthly10[metric] - monthly10[refcol]

compare_2022 = monthly10.loc[monthly10.year == 2022, [
    "date", "scene_count", TARGET_COL, "nominal_lowvv_km2_departure", "rain_mm", "rain_mm_departure", "temperature_c", "temperature_c_departure", "rain_lag1_mm", "rain_lag1_mm_departure", "antecedent_3m_rain_mm", "antecedent_3m_rain_mm_departure"
]].copy()
print("=== 2022 MONTH-MATCHED HYDROCLIMATE / RADAR DEPARTURES ===")
display(compare_2022)
compare_2022.to_csv(TABLES / "hydro10_2022_month_matched_departures.csv", index=False)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, constrained_layout=True)
axes[0].axhline(0, color="black", lw=.8)
axes[0].bar(compare_2022.date.dt.strftime("%b"), compare_2022.nominal_lowvv_km2_departure)
axes[0].set_ylabel("Low-VV minus other-year median (km²)")
axes[1].axhline(0, color="black", lw=.8)
axes[1].bar(compare_2022.date.dt.strftime("%b"), compare_2022.rain_mm_departure)
axes[1].set_ylabel("CHIRPS minus other-year median (mm)")
axes[1].set_xlabel("Month in 2022")
for ax in axes: ax.grid(axis="y", alpha=.2)
fig.suptitle("2022 month-matched departures | Descriptive comparison, not causation")
path_2022 = FIGURES / "hydro10_2022_radar_rainfall_departures.png"
fig.savefig(path_2022, dpi=230, bbox_inches="tight"); plt.show(); plt.close(fig)
print("Saved:", path_2022)

## 10E — Deseasonalized associations and threshold robustness

Compute Spearman and Pearson associations between **month-matched departures**, not just raw seasonal cycles. Antecedent lags are inspected separately (0–3 months). These are descriptive associations; **no naive pixel-level or monthly iid p-values** are reported because temporal autocorrelation, shared seasonality, and small effective year count can invalidate them.

In [ ]:
responses = {
    "lowvv_m19_km2": "Conservative −19 dB",
    "nominal_lowvv_km2": "Nominal −17 dB",
    "lowvv_m15_km2": "Permissive −15 dB"
}
predictors = {
    "rain_mm": "Current-month rainfall",
    "rain_lag1_mm": "Rainfall 1 month earlier",
    "rain_lag2_mm": "Rainfall 2 months earlier",
    "rain_lag3_mm": "Rainfall 3 months earlier",
    "antecedent_3m_rain_mm": "Sum of previous 3 months rainfall",
    "temperature_c": "Current-month air temperature"
}
association_rows=[]
for response, response_label in responses.items():
    ycol = f"{response}_departure"
    for predictor, predictor_label in predictors.items():
        xcol = f"{predictor}_departure"
        valid = monthly10["radar_ok"] & monthly10[ycol].notna() & monthly10[xcol].notna()
        xy = monthly10.loc[valid, [xcol, ycol]].astype(float)
        if len(xy) < 30 or xy[xcol].nunique() < 3 or xy[ycol].nunique() < 3:
            rho = r = np.nan
        else:
            rho=float(spearmanr(xy[xcol],xy[ycol]).statistic)
            r=float(pearsonr(xy[xcol],xy[ycol]).statistic)
        association_rows.append(dict(response=response_label,predictor=predictor_label,
            response_column=response, predictor_column=predictor,n_months=len(xy),
            spearman_rho=rho, pearson_r=r,
            status="descriptive calendar-month departures; serial dependence not corrected"))
associations = pd.DataFrame(association_rows)
associations.to_csv(TABLES / "hydro10_deseasonalized_associations.csv", index=False)
print("=== DESEASONALIZED MONTHLY ASSOCIATIONS ===")
display(associations)

nominal = associations.loc[associations.response_column == "nominal_lowvv_km2"]
fig, ax = plt.subplots(figsize=(11, 4.8), constrained_layout=True)
ax.barh(nominal.predictor.iloc[::-1], nominal.spearman_rho.iloc[::-1])
ax.axvline(0, color="black", lw=.8)
ax.set_xlabel("Spearman rho (month-matched departures)")
ax.set_title("Sentinel-1 low-VV versus hydroclimate | Exploratory, 2019–2025")
ax.set_xlim(-1,1); ax.grid(axis="x", alpha=.2)
path_corr = FIGURES / "hydro10_deseasonalized_lag_associations.png"
fig.savefig(path_corr, dpi=230, bbox_inches="tight"); plt.show(); plt.close(fig)
print("Saved:", path_corr)

## 10F — Year-held-out predictive comparison (descriptive hindcast)

Test whether adding contemporaneous and antecedent climate covariates improves **out-of-year reconstruction** beyond a simple training-years month-of-year baseline. This is **not a prospective flood forecast**: current-month CHIRPS and temperature are only known after observing the month.

- **Baseline:** median low-VV area by month from training years only.
- **Climate-enhanced:** 12 calendar-month indicators + CHIRPS rainfall at lags 0, 1, 2 + ERA5 temperature, standardized/imputed using training years only; fixed Ridge regularization.
- **Evaluation:** leave one year out at a time; no shuffled pixel/month split, and no significance claims from seven folds.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

feature_cols = ["rain_lag0_mm", "rain_lag1_mm", "rain_lag2_mm", "temperature_c"]
working = monthly10.loc[monthly10.radar_ok & monthly10[TARGET_COL].notna()].copy()
# Use common predictor-complete months to ensure fair evaluation of models on same observations.
working = working.loc[working[feature_cols].notna().all(axis=1)].copy()
if len(working) < 55 or working.year.nunique() < 6:
    raise RuntimeError("Insufficient climate-complete monthly observations for 7-year held-out comparison")

predictions = []
for held_year in sorted(working.year.unique()):
    train = working[working.year != held_year].copy()
    test = working[working.year == held_year].copy()
    if len(train) < 45 or len(test) < 6:
        raise RuntimeError(f"Too few samples in held-out year {held_year}")

    # Month-climatology baseline from training years only.
    train_month_medians = train.groupby("month")[TARGET_COL].median()
    pred_baseline = test.month.map(train_month_medians).fillna(train[TARGET_COL].median()).to_numpy(float)

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    x_train_climate = scaler.fit_transform(imputer.fit_transform(train[feature_cols]))
    x_test_climate = scaler.transform(imputer.transform(test[feature_cols]))

    # Month indicators are fixed by calendar definition; no learned month statistics.
    x_train_month = np.eye(12, dtype=float)[train.month.to_numpy(int)-1]
    x_test_month = np.eye(12, dtype=float)[test.month.to_numpy(int)-1]
    x_train = np.column_stack([x_train_month, x_train_climate])
    x_test = np.column_stack([x_test_month, x_test_climate])

    model = Ridge(alpha=20.0, fit_intercept=True)
    model.fit(x_train, train[TARGET_COL].to_numpy(float))
    pred_climate = model.predict(x_test)
    for (_, row), base, enhanced in zip(test.iterrows(), pred_baseline, pred_climate):
        predictions.append(dict(date=row.date, year=int(row.year), month=int(row.month),
           actual_lowvv_km2=float(row[TARGET_COL]), month_median_pred_km2=float(base),
           climate_ridge_pred_km2=float(enhanced),
           scene_count=int(row.scene_count), training_excludes_year=int(held_year)))

pred_df=pd.DataFrame(predictions).sort_values("date").reset_index(drop=True)
pred_df.to_csv(TABLES / "hydro10_leave_year_out_predictions.csv", index=False)

def metric_rows(frame, grouping):
    rows=[]
    actual=frame.actual_lowvv_km2.to_numpy(float)
    for name, col in [("Training-month median", "month_median_pred_km2"),
                      ("Month + rainfall/temperature Ridge", "climate_ridge_pred_km2")]:
        estimate=frame[col].to_numpy(float)
        rows.append(dict(group=grouping, model=name, n_months=len(actual),
            mae_km2=float(mean_absolute_error(actual, estimate)),
            rmse_km2=float(np.sqrt(mean_squared_error(actual, estimate))),
            mean_error_km2=float(np.mean(estimate-actual))))
    return rows

metric_list=metric_rows(pred_df,"all_years")
for yr, group in pred_df.groupby("year"):
    metric_list.extend(metric_rows(group,f"held_out_{yr}"))
metrics=pd.DataFrame(metric_list)
metrics.to_csv(TABLES / "hydro10_leave_year_out_model_metrics.csv", index=False)
print("=== ALL-YEARS HELD-OUT PERFORMANCE ===")
display(metrics[metrics.group=="all_years"])
print("\n=== YEAR-BY-YEAR PERFORMANCE ===")
display(metrics[metrics.group!="all_years"])

fig, ax = plt.subplots(figsize=(13, 5), constrained_layout=True)
ax.plot(pred_df.date, pred_df.actual_lowvv_km2, marker=".", lw=1.15, label="Observed low-VV screening area")
ax.plot(pred_df.date, pred_df.month_median_pred_km2, lw=1.0, label="Held-out month-climatology baseline")
ax.plot(pred_df.date, pred_df.climate_ridge_pred_km2, lw=1.0, label="Held-out month + climate Ridge")
ax.set_xlabel("Month"); ax.set_ylabel("Low-VV area (km²)")
ax.set_title("Year-held-out explanatory hindcast | Not future inundation prediction")
ax.grid(alpha=.2); ax.legend(loc="upper left", ncol=2)
path_cv=FIGURES / "hydro10_leave_year_out_hindcast.png"
fig.savefig(path_cv, dpi=230, bbox_inches="tight"); plt.show(); plt.close(fig)
print("Saved:", path_cv)

## 10G — Specific 2022 checks and sensitivity limitations

Check whether June 2022 is unusually high for the observed rainfall context. A year-held-out residual is **not** a causal attribution estimate: only six training years define the comparison, while changes in land use, rainfall timing, storage, radar sensing, and unmeasured predictors can affect low-VV detections. The −19/−17/−15 dB area range is threshold sensitivity, **not** a statistical confidence interval.

In [ ]:
join_2022 = monthly10.loc[monthly10.year == 2022, ["date", "scene_count", "rain_mm", "rain_mm_departure", "antecedent_3m_rain_mm", "antecedent_3m_rain_mm_departure", "temperature_c", TARGET_COL, "nominal_lowvv_km2_departure", "lowvv_m19_km2", "lowvv_m15_km2"]].merge(
    pred_df[["date", "month_median_pred_km2", "climate_ridge_pred_km2"]], on="date", how="left",validate="one_to_one")
join_2022["climate_model_residual_km2"] = join_2022[TARGET_COL] - join_2022["climate_ridge_pred_km2"]
join_2022["seasonal_model_residual_km2"] = join_2022[TARGET_COL] - join_2022["month_median_pred_km2"]
print("=== 2022 EXPLANATORY MODEL DIAGNOSTICS ===")
display(join_2022)
join_2022.to_csv(TABLES / "hydro10_2022_context_and_heldout_residuals.csv",index=False)

june=join_2022[join_2022.date.dt.month==6].iloc[0]
print("\nJune 2022 nominal low-VV area (km²):", round(float(june[TARGET_COL]),3))
print("June 2022 month-matched low-VV departure (km²):", round(float(june.nominal_lowvv_km2_departure),3))
print("June 2022 CHIRPS monthly rainfall (mm):", round(float(june.rain_mm),2))
print("June 2022 CHIRPS month-matched departure (mm):", round(float(june.rain_mm_departure),2))
print("June 2022 climate-Ridge out-of-year residual (km²):", round(float(june.climate_model_residual_km2),3) if pd.notna(june.climate_model_residual_km2) else "not available")
print("\nCAUTION: a model residual is not the effect of flooding, nor an emissions estimate.")

## 10H — Save reproducibility manifest and publication-safe findings brief

The report draws numbers **from the loaded local data**, not hard-coded research claims. It labels all radar results exploratory and provides the exact Notebook 09 processing fingerprint. Publication remains a **manual** decision; no Git commits or README edits are performed.

In [ ]:
from datetime import datetime, timezone
manifest = {
    "notebook": "10_hydroclimatic_controls_2019_2025.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "period": "2019-01 through 2025-12",
    "method_status": "exploratory_not_validated_flood_map",
    "monthly_s1_source": str(S1_CSV.relative_to(ROOT)).replace("\\", "/"),
    "hydroclimate_source": str(CLIMATE_CSV.relative_to(ROOT)).replace("\\", "/"),
    "s1_config_hash": config_hash,
    "s1_pass": parameters.get("pass_name"),
    "s1_relative_orbit": parameters.get("relative_orbit"),
    "nominal_threshold_db": -17.0,
    "rainfall_column": RAIN_COL,
    "temperature_column": TEMP_COL,
    "climate_sources": "CHIRPS daily aggregated monthly; ERA5-Land monthly aggregate (existing study CSV)",
    "n_months_radar_eligible": int(monthly10.radar_ok.sum()),
    "n_months_joint_rain_temperature": int(monthly10.joint_available.sum()),
    "year_held_out_months": int(len(pred_df)),
    "predictor_lags": [0,1,2],
    "descriptive_lags_audited": [0,1,2,3],
    "validation": "year-held-out prediction only; no independent surface-water validation",
    "causation_claimed": False,
    "methane_flux_estimated": False,
    "network_access_used": False
}
manifest_path=TABLES / "hydro10_processing_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2),encoding="utf-8")

reference_model=metrics[(metrics.group=="all_years") & (metrics.model=="Training-month median")].iloc[0]
climate_model=metrics[(metrics.group=="all_years") & (metrics.model=="Month + rainfall/temperature Ridge")].iloc[0]
june = monthly10.loc[monthly10.date == pd.Timestamp("2022-06-01")].iloc[0]

brief=f"""# Hydroclimatic context of Hakaluki Haor low-VV dynamics, 2019–2025

**Research status:** Exploratory; **not** an independently validated flood-area study.

## Data and design
- Sentinel-1 ascending orbit {parameters.get('relative_orbit')} preliminary low-VV (`VV ≤ −17 dB`) area over {int(monthly10.radar_ok.sum())} eligible months; processing fingerprint `{config_hash}`.
- CHIRPS monthly precipitation and ERA5-Land 2 m monthly temperature from the existing project CSV; {int(monthly10.joint_available.sum())} months have radar, rainfall, and temperature records.
- Calendar-month leave-one-year-out departures, lagged precipitation (0–3 months), threshold robustness (−19/−17/−15 dB), and seven held-out-year comparisons.

## Descriptive 2022 findings
- June 2022 low-VV area: **{june[TARGET_COL]:.2f} km²**, relative to other-year June month-matched median **{june.nominal_lowvv_km2_other_year_median:.2f} km²** (difference **{june.nominal_lowvv_km2_departure:+.2f} km²**).
- June 2022 CHIRPS precipitation: **{june.rain_mm:.1f} mm**; other-year June median **{june.rain_mm_other_year_median:.1f} mm** (departure **{june.rain_mm_departure:+.1f} mm**).

## Exploratory year-held-out reconstruction
- Seasonal month-median baseline: MAE **{reference_model.mae_km2:.2f} km²**, RMSE **{reference_model.rmse_km2:.2f} km²** (n={int(reference_model.n_months)} months).
- Month + rainfall/temperature Ridge: MAE **{climate_model.mae_km2:.2f} km²**, RMSE **{climate_model.rmse_km2:.2f} km²**, with the same held-out months.
- These are *year-held-out explanatory hindcasts with observed contemporaneous climate*, not forecasts. Performance may be worse than the seasonal baseline; do not assume improvement.

## Limitations
1. Low-VV screening areas have not been validated against independent inundation reference samples; flooded vegetation may remain undetected.
2. Monthly hydroclimate is spatially coarse relative to 30 m SAR mapping, and rainfall timing, water retention and terrain controls are not fully represented.
3. Only seven years are available; nearby months and mapped pixels are not independent replicates. Associations do not establish causal effects.
4. Threshold sensitivity is not a confidence interval; year-held-out skill is not flux validation.
5. No wetland methane emission flux was calculated.

## Generated figures
- `results/figures/hydro10_timeseries_rain_temperature_lowvv.png`
- `results/figures/hydro10_seasonal_cycle.png`
- `results/figures/hydro10_2022_radar_rainfall_departures.png`
- `results/figures/hydro10_deseasonalized_lag_associations.png`
- `results/figures/hydro10_leave_year_out_hindcast.png`
"""
brief_path=REPORTS / "NOTEBOOK10_HYDROCLIMATE_FINDINGS.md"
brief_path.write_text(brief, encoding="utf-8")
print("=== NOTEBOOK 10 COMPLETE ===")
print("Manifest:", manifest_path)
print("Provisional findings brief:", brief_path)
print("No Earth Engine requests, public README changes, or Git commits performed.")
print("All results remain subject to independent inundation validation.")